In [2]:
# Libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import imageio as iio
import plotly.graph_objects as go
from PIL import Image
import math

In [ ]:
# Comment unneeded files

# chemT = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/09nov14/SalishSea_1h_20141109_20141109_chem_T.nc")
# diagT = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/09nov14/SalishSea_1h_20141109_20141109_diag_T.nc")
# gridW = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/09nov14/SalishSea_1h_20141109_20141109_grid_W.nc")

nov0914 = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/09nov14/SalishSea_1h_20141109_20141109_chem_T.nc")
nov1014 = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/10nov14/SalishSea_1h_20141110_20141110_chem_T.nc")

dec1314 = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/13dec14/SalishSea_1h_20141213_20141213_chem_T.nc")
dec1414 = xr.open_dataset("/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/14dec14/SalishSea_1h_20141214_20141214_chem_T.nc")

: 

In [3]:
# Load the datasets into a list for easier handling

files = [
    # "/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/09nov14/SalishSea_1h_20141109_20141109_chem_T.nc",
    # "/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/10nov14/SalishSea_1h_20141110_20141110_chem_T.nc",
    "/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/13dec14/SalishSea_1h_20141213_20141213_chem_T.nc",
    "/ocean/atall/MOAD/Model/202410b/oxygen_NegVMixBP/14dec14/SalishSea_1h_20141214_20141214_chem_T.nc"
]

# Combined the datasets into one xarray dataset using open_mfdataset

combined = xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim="time_counter",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    join="override"
)

In [4]:
combined

<xarray.Dataset> Size: 14GB
Dimensions:                     (y: 898, x: 398, nvertex: 4, deptht: 40,
                                 axis_nbounds: 2, time_counter: 48)
Coordinates:
    nav_lat                     (y, x) float32 1MB dask.array<chunksize=(898, 398), meta=np.ndarray>
    nav_lon                     (y, x) float32 1MB dask.array<chunksize=(898, 398), meta=np.ndarray>
  * deptht                      (deptht) float32 160B 0.5 1.5 ... 414.5 441.5
  * time_counter                (time_counter) datetime64[ns] 384B 2014-12-13...
    time_centered               (time_counter) datetime64[ns] 384B dask.array<chunksize=(1,), meta=np.ndarray>
Dimensions without coordinates: y, x, nvertex, axis_nbounds
Data variables:
    bounds_nav_lon              (y, x, nvertex) float32 6MB dask.array<chunksize=(898, 398, 4), meta=np.ndarray>
    bounds_nav_lat              (y, x, nvertex) float32 6MB dask.array<chunksize=(898, 398, 4), meta=np.ndarray>
    area                        (y, x) float32 1MB dask.array<chunksize=(898, 398), meta=np.ndarray>
    deptht_bounds               (deptht, axis_nbounds) float32 320B dask.array<chunksize=(40, 2), meta=np.ndarray>
    time_centered_bounds        (time_counter, axis_nbounds) datetime64[ns] 768B dask.array<chunksize=(1, 2), meta=np.ndarray>
    time_counter_bounds         (time_counter, axis_nbounds) datetime64[ns] 768B dask.array<chunksize=(1, 2), meta=np.ndarray>
    PAR                         (time_counter, deptht, y, x) float32 3GB dask.array<chunksize=(1, 40, 898, 398), meta=np.ndarray>
    turbidity                   (time_counter, deptht, y, x) float32 3GB dask.array<chunksize=(1, 40, 898, 398), meta=np.ndarray>
    dissolved_inorganic_carbon  (time_counter, deptht, y, x) float32 3GB dask.array<chunksize=(1, 40, 898, 398), meta=np.ndarray>
    total_alkalinity            (time_counter, deptht, y, x) float32 3GB dask.array<chunksize=(1, 40, 898, 398), meta=np.ndarray>
    dissolved_oxygen            (time_counter, deptht, y, x) float32 3GB dask.array<chunksize=(1, 40, 898, 398), meta=np.ndarray>
    CO2_flux                    (time_counter, y, x) float32 69MB dask.array<chunksize=(1, 898, 398), meta=np.ndarray>
Attributes:
    name:         SalishSea_1h_20141210_20141219_chem_T
    description:  chemistry and light
    title:        chemistry and light
    Conventions:  CF-1.6
    timeStamp:    2025-Dec-02 12:29:20 GMT
    uuid:         faa34fad-1bd5-43e0-965c-ff35179e066a

In [ ]:
# Map GIF Maker

var = "dissolved_oxygen" # Variable to plot (e.g., "dissolved_oxygen", "turbidity", "PAR", etc.)
n = 22  # Index for the depth level you want to plot (e.g., 0 for surface)

#for i in range(0, combined.sizes["deptht"]):

depth = (combined["deptht"].values[n]).astype(float).round(1)  # Get the depth value for the n index (e.g. 0 - 39)
foldername = f"{var}_maps_{depth}m for 13-14 Dec 2014" # Change data according to files used in combined dataset

#-----------------------

outdir = Path(foldername)
outdir.mkdir(exist_ok=True)

for t in range(0,combined.sizes["time_counter"]):
    plt.figure(figsize=(7, 5))

    combined[var].isel(
        time_counter=t,
        deptht=n
    ).plot(
        x="x",
        y="y"
    )

    plt.title(f"{var} at {depth} m, time index {t}")
    plt.savefig(outdir / f"{var}_surface_t{t:03d}.png", dpi=150)
    plt.close()


# Folder where your 24 plots are saved
plot_dir = Path(foldername)
files = sorted(plot_dir.glob("*.png"))

imgs = [Image.open(f).convert("RGBA") for f in files]

max_width = max(img.width for img in imgs)
max_height = max(img.height for img in imgs)

frames = []

for img in imgs:
    canvas = Image.new("RGBA", (max_width, max_height), "white")

    x_offset = (max_width - img.width) // 2
    y_offset = (max_height - img.height) // 2

    canvas.paste(img, (x_offset, y_offset))
    frames.append(np.array(canvas))

iio.mimsave(
    f"{var}_{depth}m_13-14_Dec_2014.gif",
    frames,
    duration=1,
    loop=0
)

In [ ]:
# Gridded GIF Maker
gif_files = list(Path(".").glob("*.gif")) # Get all GIF files in the current directory
gif_files = sorted(Path(".").glob("*.gif")) # Sort the GIF files by name (you can customize this sorting if needed)

ngifs = len(gif_files) # Number of GIFs to combine
ncols = math.ceil(math.sqrt(ngifs)) # Number of columns in the grid (e.g., 2 for 4 GIFs, 3 for 5-9 GIFs, etc.)
nrows = math.ceil(ngifs / ncols) # Number of rows in the grid (e.g., 2 for 4 GIFs, 3 for 5-9 GIFs, etc.)

# Output file
output_gif = "combined_grid.gif" # Name of the output combined GIF file

# -----------------------
# Load all GIF frames
# -----------------------
all_gifs = []

for gif in gif_files:
    frames = iio.mimread(gif)
    pil_frames = [Image.fromarray(frame).convert("RGBA") for frame in frames]
    all_gifs.append(pil_frames)

# -----------------------
# Decide how many frames to use
# -----------------------
nframes = min(len(frames) for frames in all_gifs)

# -----------------------
# Make all frames same size
# -----------------------
max_width = 0
max_height = 0

for gif_frames in all_gifs:
    for frame in gif_frames[:nframes]:
        max_width = max(max_width, frame.width)
        max_height = max(max_height, frame.height)

# Pad each frame to the same size
for i in range(len(all_gifs)):
    padded_frames = []
    for frame in all_gifs[i][:nframes]:
        canvas = Image.new("RGBA", (max_width, max_height), "white")
        x_offset = (max_width - frame.width) // 2
        y_offset = (max_height - frame.height) // 2
        canvas.paste(frame, (x_offset, y_offset))
        padded_frames.append(canvas)
    all_gifs[i] = padded_frames

# -----------------------
# Build grid frames
# -----------------------
grid_frames = []

for t in range(nframes):
    grid_canvas = Image.new(
        "RGBA",
        (ncols * max_width, nrows * max_height),
        "white"
    )

    for k, gif_frames in enumerate(all_gifs):
        row = k // ncols
        col = k % ncols

        x = col * max_width
        y = row * max_height

        grid_canvas.paste(gif_frames[t], (x, y))

    grid_frames.append(np.array(grid_canvas))

# -----------------------
# Save the combined GIF
# -----------------------
iio.mimsave(output_gif, grid_frames, duration=1, loop=0)

print(f"Saved {output_gif}")

Saved combined_grid.gif
